# HR Employee Attrition Analysis
**Author:** Mohammed Sameer Khazi  
**Dataset:** IBM HR Analytics Employee Attrition & Performance  
**Goal:** Predict employee attrition using Random Forest Classifier

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)
from imblearn.over_sampling import SMOTE

os.makedirs('report_images', exist_ok=True)
print('Libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f'Shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Statistical Summary ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
print('=== Attrition Distribution ===')
print(df['Attrition'].value_counts())
print(f"\nAttrition Rate: {df['Attrition'].value_counts(normalize=True)['Yes']*100:.1f}%")

### Chart 1 — Attrition Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
attrition_counts = df['Attrition'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(attrition_counts.index, attrition_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Attrition Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Attrition')
axes[0].set_ylabel('Count')
for i, v in enumerate(attrition_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(attrition_counts.values, labels=attrition_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Attrition Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Employee Attrition Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('report_images/chart1_attrition_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart1_attrition_distribution.png')

### Chart 2 — Age Distribution by Attrition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
for label, color in zip(['No', 'Yes'], ['#3498db', '#e74c3c']):
    axes[0].hist(df[df['Attrition'] == label]['Age'], bins=20,
                 alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_title('Age Distribution by Attrition', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend(title='Attrition')

# Monthly Income distribution
for label, color in zip(['No', 'Yes'], ['#3498db', '#e74c3c']):
    axes[1].hist(df[df['Attrition'] == label]['MonthlyIncome'], bins=20,
                 alpha=0.6, label=label, color=color, edgecolor='white')
axes[1].set_title('Monthly Income Distribution by Attrition', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Monthly Income')
axes[1].set_ylabel('Count')
axes[1].legend(title='Attrition')

plt.tight_layout()
plt.savefig('report_images/chart2_age_income_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart2_age_income_distribution.png')

### Chart 3 — Attrition by Department & Job Role

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Department
dept_attr = df.groupby('Department')['Attrition'].value_counts(normalize=True).unstack() * 100
dept_attr.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Attrition Rate by Department', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Department')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Attrition')
axes[0].tick_params(axis='x', rotation=15)

# Job Role
role_attr = df.groupby('JobRole')['Attrition'].value_counts(normalize=True).unstack() * 100
role_attr['Yes'].sort_values().plot(kind='barh', ax=axes[1], color='#e74c3c', edgecolor='white')
axes[1].set_title('Attrition Rate by Job Role', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Attrition Rate (%)')
axes[1].set_ylabel('Job Role')

plt.tight_layout()
plt.savefig('report_images/chart3_dept_jobrole_attrition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart3_dept_jobrole_attrition.png')

### Chart 4 — Correlation Heatmap

In [ ]:
df_corr = df.copy()
df_corr['Attrition_num'] = (df_corr['Attrition'] == 'Yes').astype(int)
numeric_cols = df_corr.select_dtypes(include=[np.number]).columns.tolist()

fig, ax = plt.subplots(figsize=(16, 12))
corr_matrix = df_corr[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('report_images/chart4_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart4_correlation_heatmap.png')

### Chart 5 — WorkLife Balance & Overtime vs Attrition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# WorkLifeBalance
wlb = df.groupby('WorkLifeBalance')['Attrition'].value_counts(normalize=True).unstack() * 100
wlb.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Attrition by Work-Life Balance', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Work-Life Balance (1=Bad, 4=Best)')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Attrition')
axes[0].tick_params(axis='x', rotation=0)

# OverTime
ot = df.groupby('OverTime')['Attrition'].value_counts(normalize=True).unstack() * 100
ot.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1].set_title('Attrition by OverTime', fontsize=13, fontweight='bold')
axes[1].set_xlabel('OverTime')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Attrition')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('report_images/chart5_worklife_overtime_attrition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart5_worklife_overtime_attrition.png')

## 4. Data Cleaning & Preprocessing

In [ ]:
# Drop constant / redundant columns
cols_to_drop = ['EmployeeCount', 'StandardHours', 'Over18']
df.drop(columns=cols_to_drop, inplace=True)
print(f'Dropped columns: {cols_to_drop}')
print(f'Remaining shape: {df.shape}')

In [ ]:
# Encode categorical columns with LabelEncoder
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns to encode: {cat_cols}')

for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    
print('\nEncoding complete. All dtypes now numeric.')
df.dtypes.value_counts()

In [ ]:
# Split features and target
X = df.drop(columns=['Attrition'])
y = df['Attrition']

print(f'Features shape: {X.shape}')
print(f'Target distribution before SMOTE:\n{y.value_counts()}')

## 5. Handle Class Imbalance with SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f'Shape after SMOTE: {X_resampled.shape}')
print(f'Target distribution after SMOTE:\n{pd.Series(y_resampled).value_counts()}')

## 6. Train/Test Split & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print('Random Forest model trained successfully.')

## 7. Model Evaluation

In [ ]:
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print(f'Accuracy : {acc:.4f}')
print(f'ROC-AUC  : {roc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Attrition', 'Attrition']))

### Chart 6 — Confusion Matrix & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Attrition', 'Attrition'],
            yticklabels=['No Attrition', 'Attrition'],
            linewidths=1, linecolor='white')
axes[0].set_title(f'Confusion Matrix\n(Accuracy: {acc:.2%})', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature Importance (top 15)
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top15 = importances.nlargest(15).sort_values()
top15.plot(kind='barh', ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Top 15 Feature Importances', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('report_images/chart6_confusion_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart6_confusion_feature_importance.png')

## 8. Save Model

In [ ]:
joblib.dump(rf_model, 'model.pkl')
print('Model saved to model.pkl')

# Verify saved model
loaded = joblib.load('model.pkl')
verify_pred = loaded.predict(X_test)
print(f'Verification accuracy: {accuracy_score(y_test, verify_pred):.4f}')
print('\nAll report images saved to report_images/')
print(os.listdir('report_images'))

## 9. Model Comparison — Random Forest vs Logistic Regression vs XGBoost

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

models = {
    'Random Forest'      : rf_model,
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'XGBoost'            : XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                         use_label_encoder=False, eval_metric='logloss', random_state=42),
}

comparison = []
for name, clf in models.items():
    if name != 'Random Forest':
        clf.fit(X_train, y_train)
    y_pred  = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    comparison.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred) * 100, 2),
        'Recall'   : round(recall_score(y_test, y_pred) * 100, 2),
        'F1-Score' : round(f1_score(y_test, y_pred) * 100, 2),
        'ROC-AUC'  : round(roc_auc_score(y_test, y_proba) * 100, 2),
    })

df_comp = pd.DataFrame(comparison)
print('=== Model Comparison Table ===')
print(df_comp.to_string(index=False))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

metrics_show = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
colors       = ['#0f62fe', '#f0883e', '#bc8cff']
x = np.arange(len(metrics_show))
w = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

for i, (row, color) in enumerate(zip(comparison, colors)):
    vals = [row[m] for m in metrics_show]
    bars = ax.bar(x + i*w, vals, w, label=row['Model'],
                  color=color, edgecolor='#0d1117', linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f'{val:.1f}', ha='center', va='bottom',
                fontsize=7.5, color='white', fontweight='bold')

ax.set_xticks(x + w)
ax.set_xticklabels(metrics_show, fontsize=11, color='#e6edf3')
ax.set_yticks(range(0, 105, 10))
ax.set_yticklabels([f'{v}%' for v in range(0, 105, 10)], color='#8b949e')
ax.set_ylim(0, 108)
ax.set_ylabel('Score (%)', color='#8b949e', fontsize=11)
ax.set_title('Model Comparison — RF vs Logistic Regression vs XGBoost',
             fontsize=13, fontweight='bold', color='#e6edf3', pad=14)
ax.legend(fontsize=10, facecolor='#21262d', labelcolor='#e6edf3',
          edgecolor='#21262d', loc='lower right')
ax.grid(axis='y', color='#21262d', linewidth=0.8)
ax.spines[:].set_color('#21262d')

best = df_comp.loc[df_comp['ROC-AUC'].idxmax(), 'Model']
best_roc = df_comp['ROC-AUC'].max()
ax.text(0.02, 0.97, f'Best Model: {best}  (ROC-AUC {best_roc}%)',
        transform=ax.transAxes, ha='left', va='top',
        fontsize=10, color='#3fb950', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#0d1117',
                  edgecolor='#3fb950', linewidth=1.2))

plt.tight_layout()
plt.savefig('report_images/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: report_images/model_comparison.png')
print(f'Best model: {best}  (ROC-AUC {best_roc}%)')

## Summary

| Step | Detail |
|------|--------|
| Dataset | 1,470 employees, 35 features |
| Dropped | EmployeeCount, StandardHours, Over18 |
| Encoding | LabelEncoder on all categorical columns |
| Imbalance | Addressed with SMOTE (random_state=42) |
| Models | Random Forest (best saved), Logistic Regression, XGBoost |
| Best Model | XGBoost — Accuracy 90.08%, ROC-AUC 96.86% |
| Output | model.pkl + 7 charts in report_images/ |